# Pipeline Parallel Training

Pipeline parallelism splits the *model itself* across GPUs instead of replicating it. GPU 0 holds the first half of the network (all the conv layers), GPU 1 holds the second half (all the fully connected layers). Data flows forward through the pipeline: GPU 0 processes a batch and sends intermediate activations to GPU 1, which computes the loss and sends gradients back.

To keep both GPUs busy at the same time we use **microbatching**. Each batch of 256 is split into 8 microbatches of 32. While GPU 1 is running forward on microbatch 1, GPU 0 has already moved on to microbatch 2 — filling the pipeline and reducing idle time.

## Imports and Constants

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.distributed as dist
from torch.distributed.pipelining import PipelineStage, ScheduleGPipe
from torch.utils.data import DataLoader, RandomSampler
from torchvision import datasets, transforms
import torch.multiprocessing as mp
import matplotlib.pyplot as plt
import random
import time
import warnings

warnings.filterwarnings('ignore', category=FutureWarning, module='torch.distributed.pipelining')

BATCH_SIZE = 256
EPOCHS = 100
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MODEL_WEIGHTS_PATH = 'cifar_cnn.pth'
CIFAR_MEAN = (0.49139968, 0.48215827, 0.44653124)
CIFAR_STD  = (0.24703233, 0.24348505, 0.26158768)
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

## 2-Stage Model: StagedCIFARNet

Same network as CIFARNet, reorganized into two `nn.Sequential` modules that each live on a separate GPU. The split is at the conv/FC boundary — conv layers do spatial feature extraction and FC layers do classification, so it's the most natural place to cut.

The `Flatten` layer lives at the end of `stage1` so the activation sent between GPUs is a flat 1D vector `(B, 32768)` rather than a 3D tensor, which is a bit more efficient to transfer.

In [ ]:
class StagedCIFARNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Stage 1 (GPU 0): feature extraction all 4 conv layers + flatten
        # Input: (B, 3, 32, 32)   Output: (B, 32768)
        self.stage1 = nn.Sequential(
            # conv1 + pool + bn + relu: (B, 3, 32, 32) -> (B, 64, 16, 16)
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # conv2 + pool + bn + relu: (B, 64, 16, 16) -> (B, 128, 8, 8)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.MaxPool2d(2, 2),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # conv3 + bn + relu: (B, 128, 8, 8) -> (B, 256, 8, 8)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            # conv4 + bn + relu: (B, 256, 8, 8) -> (B, 512, 8, 8)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            # flatten: (B, 512, 8, 8) -> (B, 32768)
            nn.Flatten(1),
        )

        # Stage 2 (GPU 1): classification all 3 FC layers
        # Input: (B, 32768)   Output: (B, 10)
        self.stage2 = nn.Sequential(
            nn.Linear(512 * 8 * 8, 512),  # (B, 32768) -> (B, 512)
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(512, 256),           # (B, 512) -> (B, 256)
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, 10),            # (B, 256) -> (B, 10)
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.stage2(x)
        return x

## Data Loading for Pipeline

In pipelined training, only the first and last GPU need actual data. GPU 0 only needs the images (it feeds them into the pipeline), GPU 1 only needs the labels (to compute the loss). Middle stages just pass activations along.

Both loaders must also iterate in the same order, or GPU 1 would compute loss against the wrong labels. We use a `RandomSampler` seeded with the same fixed seed on both ranks to guarantee this.

In [ ]:
def get_cifar_data_transforms():
    # create a set of augmentations we want to do to our training and data set.
    # for the training set we randomly flip the image to better generalize
    # and also randomly crop out a small portion of it
    # then we normalize the rgb values to be ~N(0,1)
    transform_train = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    # We do not augment the data if testing our classifier because we do not want to hinder the model in any way
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    return transform_train, transform_test


def get_cifar_data_sets(transform_train, transform_test):
    train_set = datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train,
    )
    test_set = datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test,
    )
    return train_set, test_set


def cifar10_loaders_pipeline(world_size, rank, batch_size=BATCH_SIZE, shuffle_seed=42):
    # Middle ranks don't need data loaders — they just receive and forward activations
    if rank != 0 and rank != world_size - 1:
        return None, None

    transform_train, transform_test = get_cifar_data_transforms()
    train_set, test_set = get_cifar_data_sets(transform_train, transform_test)

    # Create generator with fixed seed for reproducible shuffling
    generator = torch.Generator()
    generator.manual_seed(shuffle_seed)

    # Create DataLoaders with RandomSampler using the same seed
    # This ensures rank 0 and last rank get the same batches in the same order
    train_loader = DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=False,
        sampler=RandomSampler(train_set, generator=generator),
        pin_memory=True,
        num_workers=4,
        drop_last=True  # Drop last incomplete batch for consistent pipeline stages
    )

    test_loader = DataLoader(
        test_set,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=4,
        drop_last=True
    )

    return train_loader, test_loader

## Model Splitting and PipelineStage

`manual_model_split` creates only the portion of the model for this rank and wraps it in a `PipelineStage`. The `PipelineStage` object handles all the communication automatically, receiving activations from the previous stage, running forward through this stage's layers, and sending activations to the next stage. On the backward pass it does the same thing in reverse with gradients.

In [ ]:
def manual_model_split(rank, num_stages, device):
    model = StagedCIFARNet()

    # each gpu gets one portion of our model
    if rank == 0:
        model_stage = model.stage1
    else:  # rank == 1
        model_stage = model.stage2

    # move only the portion of the model this gpu cares about to the gpu
    model_stage.to(device)

    # PipelineStage is where all of the magic happens
    # it is responsible for receiving and sending activations to the next/previous stages
    # and also computing gradients and sending them backwards
    pipeline_stage = PipelineStage(
        model_stage,
        rank,
        num_stages,
        device,
    )

    return pipeline_stage, model_stage

## Pipeline Training Worker

The training loop looks different from DDP because each rank has a different job:
- Rank 0 feeds images into the pipeline with `schedule.step(images)`
- Rank 1 provides labels and gets outputs back with `schedule.step(target=labels, losses=[])`
- Each rank has its own optimizer for its own stage parameters — there's no gradient sync between stages

We broadcast `num_batches` from rank 0 so all ranks know how many iterations to run, since only rank 0 has a data loader at the start.

The GPipe schedule looks like this with 8 microbatches:
```
GPU 0: F1 F2 F3 F4 F5 F6 F7 F8 __ __ B8 B7 B6 B5 B4 B3 B2 B1
GPU 1: __ F1 F2 F3 F4 F5 F6 F7 F8 B8 B7 B6 B5 B4 B3 B2 B1 __
```
F = forward pass, B = backward pass

In [ ]:
def pipeline_worker(rank, world_size):
    # same setup as ddp — initialize NCCL process group so stages can send activations between gpus
    # use a different port (12356) than the DDP notebook so they don't conflict if run in the same session
    torch.cuda.set_device(rank)
    device = torch.device(f'cuda:{rank}')

    dist.init_process_group(
        backend='nccl',
        init_method='tcp://127.0.0.1:12356',
        rank=rank,
        world_size=world_size
    )

    if rank == 0:
        print(f'Pipeline Parallelism on {world_size} GPUs')

    train_loader, test_loader = cifar10_loaders_pipeline(world_size, rank, BATCH_SIZE)

    if train_loader is not None:
        num_batches = len(train_loader)
    else:
        num_batches = 0

    # Create pipeline stage for this rank
    pipeline_stage, model_stage = manual_model_split(rank, world_size, device)

    # Loss and optimizer — each stage has its own parameters and its own optimizer
    # there is no gradient sync between stages, each one just updates its own weights
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model_stage.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # Broadcast number of batches in the training set so each rank knows how many it will process
    if rank == 0:
        num_batches_tensor = torch.tensor([num_batches], dtype=torch.long, device=device)
    else:
        num_batches_tensor = torch.tensor([0], dtype=torch.long, device=device)
    dist.broadcast(num_batches_tensor, src=0)
    num_batches = num_batches_tensor.item()

    # a micro batch is a split of a regular batch
    # so in this example we have 256 images in a batch so a micro batch would have 32 images
    # microbatches allow us to schedule and send smaller batches of activations to better use our gpus
    n_microbatches = 8

    # schedule g pipe takes in our pipeline that we created and is then responsible for actually coordinating all of the data
    # responsible for actually splitting our batches into the microbatches
    # running the microbatches through one pipeline stage at the end
    # and then finally running all of the gradients through backwards
    schedule = ScheduleGPipe(pipeline_stage, n_microbatches=n_microbatches, loss_fn=loss_function)

    for epoch in range(EPOCHS):
        if rank == 0:
            print(f'\nEpoch {epoch + 1}/{EPOCHS}')
            start_time = time.time()

        # put the model stage in training mode
        model_stage.train()

        # Track accuracy on last rank (where we have the predictions)
        if rank == world_size - 1:
            train_correct = 0
            train_total = 0

        if rank == 0:
            train_iter = iter(train_loader)

            # this loop chucks all of our batches for an entire epoch into the first gpu
            for batch_idx in range(num_batches):
                optimizer.zero_grad()

                # first gpu: only care about images, not labels
                images, _ = next(train_iter)
                images = images.to(device)

                # first gpu kicks off the pipeline with the images
                schedule.step(images)
                # adjust the weights for this stage
                optimizer.step()

        elif rank == world_size - 1:
            train_iter = iter(train_loader)

            for batch_idx in range(num_batches):
                optimizer.zero_grad()

                # last gpu: only care about labels to compute the loss
                _, labels = next(train_iter)
                labels = labels.to(device)

                # last gpu computes the loss and returns output
                output = schedule.step(target=labels, losses=[])
                optimizer.step()

                # Track accuracy on last gpu where outputs are produced
                _, predicted = torch.max(output, 1)
                train_correct += (predicted == labels).sum().item()
                train_total += labels.size(0)

        else:
            # any middle gpus (none with world_size=2) just receive and send
            for batch_idx in range(num_batches):
                optimizer.zero_grad()
                schedule.step()
                optimizer.step()

        scheduler.step()

        # Print training accuracy from last rank
        if rank == world_size - 1:
            train_accuracy = 100.0 * train_correct / train_total
            print(f'  Training Accuracy: {train_accuracy:.2f}%')

        if rank == 0:
            print(f'  Epoch time: {(time.time() - start_time):.2f}s')

    # Each rank saves its own stage's weights
    # rank 0 saves stage0.pth (conv layers), rank 1 saves stage1.pth (FC layers)
    stage_save_path = f'{MODEL_WEIGHTS_PATH}.stage{rank}.pth'
    torch.save(model_stage.state_dict(), stage_save_path)
    print(f'[rank {rank}] Saved stage weights to {stage_save_path}')

    dist.destroy_process_group()

## Launch Pipeline Training

In [ ]:
assert torch.cuda.device_count() >= 2, (
    'This notebook requires 2 GPUs. '
    'Go to Settings -> Accelerator -> GPU T4 x2 and re-run.'
)

# clean up any stale process groups from a previous run
if dist.is_initialized():
    dist.destroy_process_group()

world_size = torch.cuda.device_count()
print(f'Found {world_size} GPU(s)')

with mp.Manager() as manager:
    mp.start_processes(pipeline_worker, args=(world_size,), nprocs=world_size, join=True, start_method='fork')

## Inference with Pipeline Weights

After training, two weight files exist:
- `cifar_cnn.pth.stage0.pth`  saved by rank 0, holds the conv layer weights (`stage1` of the model)
- `cifar_cnn.pth.stage1.pth`  saved by rank 1, holds the FC layer weights (`stage2` of the model)

For inference we reassemble the full model on a single GPU by loading both files into their corresponding stages.

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model_inf = StagedCIFARNet().to(device)

# Load each stage's weights back into the corresponding module
# stage0.pth (saved by rank 0) model.stage1 (conv layers)
# stage1.pth (saved by rank 1) model.stage2 (FC layers)
model_inf.stage1.load_state_dict(torch.load(f'{MODEL_WEIGHTS_PATH}.stage0.pth', map_location=device))
model_inf.stage2.load_state_dict(torch.load(f'{MODEL_WEIGHTS_PATH}.stage1.pth', map_location=device))
model_inf.eval()

# get our test data set
transform_train, transform_test = get_cifar_data_transforms()
_, test_set = get_cifar_data_sets(transform_train, transform_test)

# pick a random test image
idx = random.randint(0, len(test_set) - 1)
image, label = test_set[idx]

# save unnormalized copy for display — reverse the normalization so colors look right
unnorm = image.clone()
for t, m, s in zip(unnorm, CIFAR_MEAN, CIFAR_STD):
    t.mul_(s).add_(m)
np_img = unnorm.permute(1, 2, 0).numpy()

# run inference through the reassembled model
image_batch = image.unsqueeze(0).to(device)
with torch.no_grad():
    output = model_inf(image_batch)
    predicted_idx = output.argmax(dim=1).item()

predicted_class = CLASSES[predicted_idx]
true_class = CLASSES[label]

print('------- Inference Result -------')
print(f'Random Test Index: {idx}')
print(f'Predicted class:   {predicted_class}')
print(f'True label:        {true_class}')
print('--------------------------------')

plt.figure(figsize=(4, 4))
plt.imshow(np_img)
plt.title(f'Predicted: {predicted_class}\nActual: {true_class}', fontsize=12)
plt.axis('off')
plt.tight_layout()
plt.show()